# Model Régression Logistique

## Import

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import pandas as pd





## Explication

Pour ce model nous repondrons a la question numero 3 : `Which product have the heights probability of being reordered`.

Pour ce faire nous allons nous baser sur un mode de Regression Logistique, qui fait parti des model des algorithmes de classification.



j'ai tester de faire avec user_id ety order_id dans le groupby

#### Documentation technique :
* .reset_index() --> permet lors d'un groupby de faire des elements des columns

# Code

## Lecture & Creation des DF

### Lecture CSV clean

In [2]:
#ici c'est mon CSV de base
CSV_untouch = pd.read_csv('CSV_Cleaned.csv')

In [5]:
print(CSV_untouch)

         department                aisle                    product_name  \
0            frozen  frozen meat seafood              Ground Turkey Chub   
1            frozen  frozen meat seafood              Ground Turkey Chub   
2            frozen  frozen meat seafood              Ground Turkey Chub   
3            frozen  frozen meat seafood              Ground Turkey Chub   
4            frozen  frozen meat seafood              Ground Turkey Chub   
...             ...                  ...                             ...   
13690535    missing              missing  Lemon Cayenne Drinking Vinegar   
13690536    missing              missing  Lemon Cayenne Drinking Vinegar   
13690537    missing              missing  Lemon Cayenne Drinking Vinegar   
13690538    missing              missing  Lemon Cayenne Drinking Vinegar   
13690539    missing              missing                K Cup Dark Blend   

          order_id   user_id  order_number  order_dow  order_hour_of_day  \
0          

### Creation du DF d'entrainement

In [3]:
#on va faire un DF avec uniquement les product_name et les user_id pour avoir une ligne par produit et autant le lignes que de produits acheter par un user_id
CSV_Clean = CSV_untouch.groupby(["user_id", "product_name"]).agg({
    "order_number": 'max',
    "add_to_cart_order": 'mean'
}).reset_index() 



### Creation du DF reponse

In [4]:
#ici on va faire nos reponses donc on garde le meme ordre et on va juste prendre la moyenne de reordered
CSV_TP = CSV_untouch.groupby(["user_id", "product_name"]).agg({
    "reordered": 'mean'
}).reset_index() 


### Conformite des valeurs et column

In [ ]:
# comme on est sur un model de LogisticRegression on doit donner soit des 0 soit 1 donc on coupe le reordored en 0 ou 1
CSV_TP['target'] = (CSV_TP['reordered'] > 0.5).astype(int)

# ici on va merge nos deux DF, celui nettiyer qui contient nos column d'entrainement + celui qui contient nos reponse avec les 0 ou 1 
df_final = pd.merge(CSV_Clean, CSV_TP[['user_id', 'product_name', 'target']], on=['user_id', 'product_name'])
# on degage les vide 
df_final = df_final.dropna()


### Definition des valeurs

In [ ]:
# on defini ici nos column issue du DF finale
X = df_final[['order_number', 'add_to_cart_order']]
y = df_final['target']

# on va separer nos data en 2, 70% pour 'apprendre et 30% pour le tester comme ca on a un meilleur resultat
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


## Verification de conformite

In [61]:
# verification pou s'assurer qu'on a pas de probleme de nombre du style pas le meme nombre de row ou de column vide
print("--- NaNs dans les Features ---")
print(col_predict.isna().sum())

print("\n--- NaNs dans la Cible ---")
print(CSV_TP['target'].isna().sum())

# verifie si les index correspondent
print(f"\nNombre de lignes col_predict : {len(col_predict)}")
print(f"Nombre de lignes target : {len(CSV_TP)}")

--- NaNs dans les Features ---
order_number         0
add_to_cart_order    0
dtype: int64

--- NaNs dans la Cible ---
0

Nombre de lignes col_predict : 7656563
Nombre de lignes target : 7656563


### Entrenement du model

In [ ]:
# ic on entraine l'IA avec notre model
clf = LogisticRegression(random_state=0).fit(X_train, y_train)


## Resultat

### Precision du model

In [ ]:
# ici on va afficher la precision de notre model en pourcentage avec le truc du sheitan en bas
score = clf.score(X_test, y_test)
print(f"Précision du modèle : {score * 100:.2f}%")

### Affichage du Top

In [58]:
# ici on creer une column dans notre df finale, predict_proba sert a avoir un vrqai pourcentage et non un 0 ou 1, on prend ici la column 1 car c'est les produits a 1 qui seront rachete
df_final['proba_reachat'] = clf.predict_proba(X)[:, 1]

# on regroupe nos nom de produit avec la moyenne de toute les probabilite de rachats pour chaque ites, exemple : si pomme revient 2 fois, une avec 100% de chance de rachat et l'autre 0% la moyenne fera donc 50% de chance de rachat pour les pommes
top_produits = df_final.groupby('product_name')['proba_reachat'].mean().reset_index()

# la on va sorted pour avoir en premier les plus grosse proba de rachat
top_produits = top_produits.sort_values(by='proba_reachat', ascending=False)

# on affiche le top 10 des plus grosse proba de rachat
print(top_produits.head(10))

                                            product_name  proba_reachat
36912                                Raw Veggie Wrappers       0.940422
46653                              Variety Pack Cat Food       0.935208
37552             RepleniSH Contact Lens Rewetting Drops       0.923770
13287  Duos Natural Rotisserie Chicken Flavor Cat Treats       0.922860
37790                       Rich Maple Toastables Quinoa       0.912356
31885                         Organic Vanilla Hemp Shake       0.909500
26631                               Natural Laxative Tea       0.899418
48323                    Wild Alaskan Salmon Oil 1000 Mg       0.882340
39968  Simply Strawberry Banana & Apple Freeze Dried ...       0.872986
2911     Artisan Thin Pizza Crust Rustic White Flatbread       0.868105


## Amelioration

Nous avons un model fonctionnel mais son pourcentage de precision est encore trop peu eleve, nous allons essayer de l'augmenter en modifier certains parametres.

Le code sera indentique et depourvu de commentaire car il est identique a celui precedent, seul la partie modifier sera explique et documenter. SI une question perciste remonter plus haut ou referer vous vers la documentation du project.

`/!\ Attention l'execution d'une cellule peut prendre un certains temps, une fois execute attendez simplement la fin de son execution /!\`

### Modification de `CSV_TP['target'] = (CSV_TP['reordered'] > 0.5).astype(int)`

In [ ]:
# Nous allons ici essayer plusieurs valeurs et voir la quelle pourrait nous fournir une precision plus importante.

#  nous definissons plusieurs valeurs, sachant que celle de base est 0.5
Lst_Valeur = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
# nous allons definir ici un dictionnaire qui va contenir toute nos valeurs ci dessus + leurs score de precision obtenu
Score_predict = {}

for valeur in Lst_Valeur:
    # on change ici en prennant notre valeur donc un element de notre liste Lst_Valeur
    CSV_TP['target'] = (CSV_TP['reordered'] > valeur).astype(int)

    # Code simillaire a l'exemple du dessus --------------------------------------------------------------------
    df_final = pd.merge(CSV_Clean, CSV_TP[['user_id', 'product_name', 'target']], on=['user_id', 'product_name'])
    df_final = df_final.dropna()
    X = df_final[['order_number', 'add_to_cart_order']]
    y = df_final['target']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    clf = LogisticRegression(random_state=0).fit(X_train, y_train)
    score = clf.score(X_test, y_test)
    # ----------------------------------------------------------------------------------------------------------

    # on stocke nos valeurs dans le dictionnaire
    Score_predict[valeur] = score

# on affiche notre dictionnaire
print(Score_predict)

# OUTPUT --> {0.1: 0.6374060773131897, 0.2: 0.6374060773131897, 0.3: 0.6374060773131897, 0.4: 0.6374060773131897, 0.5: 0.6764161814983136, 0.6: 0.6764161814983136, 0.7: 0.6982214387743152, 0.8: 0.7162125392201636, 0.9: 0.7269749831190582}

### Analyse

On remarque que plus on met une valeur eleve plus la precision est bonne, en realite c'est le contraire, plus nous mettons un score eleve plus l'IA ne vas prendre que des valeurs ou elle est sur que le produit est reacheter, l'IA devient donc paraisseuse et ne cherche plus a reellement comprendre.

Les resultats sont neanmoins tous dans la meme plage de valeurs et ne differe pas enormement, 0.5 est un bon compromis dans notre exemple.
(Il sera peut etre judiciteux de le bouger apres d'autres test mais il restera aux alentour des 0.4 : 0.6)

### Modifictions `train_test_split(X, y, test_size=0.3, random_state=42)`

In [ ]:
# Nous allons ici modifier le % de separation des donnes d'entrainement et de test
#Actuellement 70 % des donnes servent a entrainer notre model et 30% a les tester

# Nous allons reprendre la meme structure que precedement

#  nous definissons plusieurs valeurs, sachant que celle de base est 0.5
Lst_Valeur = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
# nous allons definir ici un dictionnaire qui va contenir toute nos valeurs ci dessus + leurs score de precision obtenu
Score_predict = {}

for valeur in Lst_Valeur:
    # Code identique a l'exemple -------------------------------------------------------------------------------
    CSV_TP['target'] = (CSV_TP['reordered'] > 0.5).astype(int)
    df_final = pd.merge(CSV_Clean, CSV_TP[['user_id', 'product_name', 'target']], on=['user_id', 'product_name'])
    df_final = df_final.dropna()
    X = df_final[['order_number', 'add_to_cart_order']]
    y = df_final['target']
    #------------------------------------------------------------------------------------------------------------
    # on va ici modifier notre `test_size` pour voir si la precision peut etre ameliorer
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=valeur, random_state=42)

    
    clf = LogisticRegression(random_state=0).fit(X_train, y_train)
    score = clf.score(X_test, y_test)
    Score_predict[valeur] = score

print(Score_predict)

# OUTPUT --> {0.1: 0.6761108433671996, 0.2: 0.6762614827928712, 0.3: 0.6764161814983136, 0.4: 0.6764844940257152, 0.5: 0.6766910065663919, 0.6: 0.6766425667912802, 0.7: 0.6765513065819339, 0.8: 0.6765657439997153, 0.9: 0.6764897567185277}

### Analyse

Nous voyons donc ici que les valeurs de precisions sont quasiment identiques entre elles, le fait de couper a different pourcentage nos donnes ne donnent pas forcement de meilleur resultats.

Nous allons donc nous mettre sur 60 % pour avoir une separation optimale

### Changement des columns d'entrainement

In [1]:
# Nous allons changer les columns d'entrainement en en ajoutant de nouvelles pour voir si la precisions augmente

# dfClean2 = pd.get_dummies(
#     CSV_untouch,
#     columns=["aisle", "department"],
#     drop_first=True,
#     dtype=int,
# )
# Colonnes_Aisle = [col for col in dfClean2.columns if col.startswith("aisle_")]
# Colonnes_Departement = [col for col in dfClean2.columns if col.startswith("department_")]

# print(Colonnes_Aisle, Colonnes_Departement)

# On va prendre depuis le CSV_untouch : 'order_dow'
# On va egalement changer l'aspect du groupy pour que ce soit plus simple de creeer des column dedans
CSV_Clean = CSV_untouch.groupby(["user_id", "product_name"]).agg(
    order_number=('order_number', 'max'),
    add_to_cart_order=('add_to_cart_order', 'mean'),

    order_hour_of_day1=('order_hour_of_day', 'mean'),
    
    
    # on va ici se servire des jours de la semaine mais on va les pousser tres loin, on prends nos jours de la semaine on va prendre,
    #la moyenne donc en moyenne quel jour de la semaine c'est achete,
    dow_mean=('order_dow', 'mean'),
    # le count c'est la frequence d'achat donc combien il a achete de X items cette semaine
    order_count=('order_dow', 'count'),
    # std c'est l'equart type donc pour savoir si il achete souvent comme d'ahbitude si c'est le cas ca sera proche de 0 sinon ca augmente
    dow_std=('order_dow', 'std')
).reset_index() 


CSV_TP['target'] = (CSV_TP['reordered'] > 0.5).astype(int)

df_final = pd.merge(CSV_Clean, CSV_TP[['user_id', 'product_name', 'target', Colonnes_Aisle, Colonnes_Departement]], on=['user_id', 'product_name'])
df_final = df_final.dropna()

# On change donc ici les column a selectionner pour l'entraienement
X = df_final[['order_number', 'add_to_cart_order', 'dow_mean', 'order_count', 'dow_std', 'order_hour_of_day1', Colonnes_Departement, Colonnes_Aisle]]


y = df_final['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.6, random_state=42)
clf = LogisticRegression(random_state=0).fit(X_train, y_train)
score = clf.score(X_test, y_test)


print(score)
OUTPUT --> 0.8293161031524151


NameError: name 'pd' is not defined

### Analyse